In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from hashlib import sha256
import json
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr, t


# ============================================================
# 1. Configuration
# ============================================================

# These are the exact filenames supplied for the clean rerun.
SUPPORT_FILENAME = "1.xlsx"
COOPERATION_FILENAME = "main file.csv"

# Every generated result and visualization goes here.
OUTPUT_DIR = Path("output 3")

WB_API_URL = (
    "https://api.worldbank.org/v2/country"
    "?format=json&per_page=400"
)

WB_SNAPSHOT_FILE = (
    OUTPUT_DIR
    / "world_bank_country_characteristics_snapshot.csv"
)


def find_input(filename: str) -> Path:
    """Find an unchanged input filename in common notebook/repository locations."""
    script_dir = (
        Path(__file__).resolve().parent
        if "__file__" in globals()
        else Path.cwd()
    )

    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "upload" / filename,
        script_dir / filename,
        script_dir / "upload" / filename,
        script_dir.parent / filename,
        script_dir.parent / "upload" / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    searched = "\n  - ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Searched:\n  - {searched}"
    )


def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as file_handle:
        for block in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# ============================================================
# 2. Complete UN-member universe
# ============================================================

# The 193 UN member states in consistent short-name form.
# India is removed below because India is the focal country.
UN_MEMBER_STATES = (
    "Afghanistan", "Albania", "Algeria", "Andorra", "Angola",
    "Antigua and Barbuda", "Argentina", "Armenia", "Australia", "Austria",
    "Azerbaijan", "Bahamas", "Bahrain", "Bangladesh", "Barbados", "Belarus",
    "Belgium", "Belize", "Benin", "Bhutan", "Bolivia",
    "Bosnia and Herzegovina", "Botswana", "Brazil", "Brunei", "Bulgaria",
    "Burkina Faso", "Burundi", "Cabo Verde", "Cambodia", "Cameroon", "Canada",
    "Central African Republic", "Chad", "Chile", "China", "Colombia", "Comoros",
    "Republic of the Congo", "Costa Rica", "Côte d'Ivoire", "Croatia", "Cuba",
    "Cyprus", "Czechia", "North Korea", "DR Congo", "Denmark", "Djibouti",
    "Dominica", "Dominican Republic", "Ecuador", "Egypt", "El Salvador",
    "Equatorial Guinea", "Eritrea", "Estonia", "Eswatini", "Ethiopia", "Fiji",
    "Finland", "France", "Gabon", "Gambia", "Georgia", "Germany", "Ghana",
    "Greece", "Grenada", "Guatemala", "Guinea", "Guinea-Bissau", "Guyana",
    "Haiti", "Honduras", "Hungary", "Iceland", "India", "Indonesia", "Iran",
    "Iraq", "Ireland", "Israel", "Italy", "Jamaica", "Japan", "Jordan",
    "Kazakhstan", "Kenya", "Kiribati", "Kuwait", "Kyrgyzstan", "Laos", "Latvia",
    "Lebanon", "Lesotho", "Liberia", "Libya", "Liechtenstein", "Lithuania",
    "Luxembourg", "Madagascar", "Malawi", "Malaysia", "Maldives", "Mali",
    "Malta", "Marshall Islands", "Mauritania", "Mauritius", "Mexico",
    "Micronesia", "Moldova", "Monaco", "Mongolia", "Montenegro", "Morocco",
    "Mozambique", "Myanmar", "Namibia", "Nauru", "Nepal", "Netherlands",
    "New Zealand", "Nicaragua", "Niger", "Nigeria", "North Macedonia", "Norway",
    "Oman", "Pakistan", "Palau", "Panama", "Papua New Guinea", "Paraguay",
    "Peru", "Philippines", "Poland", "Portugal", "Qatar", "South Korea",
    "Romania", "Russia", "Rwanda", "Saint Kitts and Nevis", "Saint Lucia",
    "Saint Vincent and the Grenadines", "Samoa", "San Marino",
    "Sao Tome and Principe", "Saudi Arabia", "Senegal", "Serbia", "Seychelles",
    "Sierra Leone", "Singapore", "Slovakia", "Slovenia", "Solomon Islands",
    "Somalia", "South Africa", "South Sudan", "Spain", "Sri Lanka", "Sudan",
    "Suriname", "Sweden", "Switzerland", "Syria", "Tajikistan", "Tanzania",
    "Thailand", "Timor-Leste", "Togo", "Tonga", "Trinidad and Tobago", "Tunisia",
    "Türkiye", "Turkmenistan", "Tuvalu", "Uganda", "Ukraine",
    "United Arab Emirates", "United Kingdom", "United States of America", "Uruguay",
    "Uzbekistan", "Vanuatu", "Venezuela", "Vietnam", "Yemen", "Zambia", "Zimbabwe",
)

COUNTRY_ALIASES = {
    "Brunei Darussalam": "Brunei",
    "Congo": "Republic of the Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Congo, Dem. Rep.": "DR Congo",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Czech Republic": "Czechia",
    "Democratic People's Republic of Korea": "North Korea",
    "Democratic Republic of the Congo": "DR Congo",
    "Federated States of Micronesia": "Micronesia",
    "Iran (Islamic Republic of)": "Iran",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Lao People's Democratic Republic": "Laos",
    "Micronesia, Federated States of": "Micronesia",
    "Principality of Liechtenstein": "Liechtenstein",
    "Republic of Korea": "South Korea",
    "Republic of Moldova": "Moldova",
    "Russian Federation": "Russia",
    "Syrian Arab Republic": "Syria",
    "Turkey": "Türkiye",
    "United Republic of Tanzania": "Tanzania",
    "United States": "United States of America",
    "Viet Nam": "Vietnam",
}


def normalize_country(value: object) -> str:
    name = str(value).strip()
    return COUNTRY_ALIASES.get(name, name)


if len(UN_MEMBER_STATES) != 193:
    raise AssertionError(
        f"Expected 193 UN members, found {len(UN_MEMBER_STATES)}."
    )

if len(set(UN_MEMBER_STATES)) != 193:
    raise AssertionError("The UN-member list contains duplicates.")

universe = sorted(set(UN_MEMBER_STATES) - {"India"})
universe_set = set(universe)

if len(universe) != 192:
    raise AssertionError(
        f"Expected 192 non-India UN members, found {len(universe)}."
    )


# ============================================================
# 3. Read and validate the two fresh inputs
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

support_path = find_input(SUPPORT_FILENAME)
cooperation_path = find_input(COOPERATION_FILENAME)

support = pd.read_excel(
    support_path,
    sheet_name="Explicit Support",
)

cooperation = pd.read_csv(
    cooperation_path,
    encoding="utf-8",
)

required_support_columns = {"Country Name", "Source"}
required_cooperation_columns = {
    "Country",
    "Area of Cooperation",
    "Area_Normalized",
}

if not required_support_columns.issubset(support.columns):
    raise ValueError(
        "Support workbook must contain: "
        + ", ".join(sorted(required_support_columns))
    )

if not required_cooperation_columns.issubset(cooperation.columns):
    raise ValueError(
        "Cooperation file must contain: "
        + ", ".join(sorted(required_cooperation_columns))
    )

if support[list(required_support_columns)].isna().any().any():
    raise ValueError("Support workbook contains missing analytical values.")

if cooperation[list(required_cooperation_columns)].isna().any().any():
    raise ValueError("Cooperation file contains missing analytical values.")

if support.duplicated().any():
    raise ValueError("Support workbook contains exactly duplicated rows.")

if cooperation.duplicated().any():
    raise ValueError("Cooperation file contains exactly duplicated rows.")

support = support.copy()
cooperation = cooperation.copy()

support["Country_Analysis"] = (
    support["Country Name"].map(normalize_country)
)

cooperation["Country_Analysis"] = (
    cooperation["Country"].map(normalize_country)
)

cooperation["Area of Cooperation"] = (
    cooperation["Area of Cooperation"]
    .astype(str)
    .str.strip()
)

cooperation["Area_Normalized"] = (
    cooperation["Area_Normalized"]
    .astype(str)
    .str.strip()
)

if support["Country_Analysis"].duplicated().any():
    duplicates = sorted(
        support.loc[
            support["Country_Analysis"].duplicated(keep=False),
            "Country_Analysis",
        ].unique()
    )
    raise ValueError(
        "Duplicate support countries after normalization: "
        + ", ".join(duplicates)
    )

support_outside_universe = sorted(
    set(support["Country_Analysis"]) - universe_set
)

if support_outside_universe:
    raise ValueError(
        "Support countries not matched to the UN-member universe: "
        + ", ".join(support_outside_universe)
    )

valid_cooperation = cooperation.loc[
    cooperation["Country_Analysis"].isin(universe_set)
].copy()

excluded_cooperation = cooperation.loc[
    ~cooperation["Country_Analysis"].isin(universe_set)
].copy()

cooperation_summary = (
    valid_cooperation
    .groupby("Country_Analysis", as_index=False)
    .agg(
        Cooperation_Entries=("Area of Cooperation", "size"),
        Cooperation_Breadth=("Area_Normalized", "nunique"),
    )
)

support_lookup = (
    support[["Country_Analysis", "Source"]]
    .rename(columns={"Source": "Support_Source"})
)

master = pd.DataFrame({"Country_Analysis": universe})

master = master.merge(
    cooperation_summary,
    on="Country_Analysis",
    how="left",
    validate="one_to_one",
)

master = master.merge(
    support_lookup,
    on="Country_Analysis",
    how="left",
    validate="one_to_one",
)

# A country absent from the cooperation file has zero recorded
# cooperation, not a missing observation.
master[["Cooperation_Entries", "Cooperation_Breadth"]] = (
    master[["Cooperation_Entries", "Cooperation_Breadth"]]
    .fillna(0)
    .astype(int)
)

master["Documented_Support"] = (
    master["Support_Source"].notna().astype(int)
)

master["Support_Source"] = master["Support_Source"].fillna("")

master["Support_Status"] = np.where(
    master["Documented_Support"].eq(1),
    "Documented explicit support",
    "No documented explicit support in supplied file",
)

if len(master) != 192:
    raise AssertionError(f"Expected 192 countries, found {len(master)}.")

if master["Documented_Support"].sum() != support["Country_Analysis"].nunique():
    raise AssertionError("Not every supporter was retained exactly once.")

if master["Cooperation_Entries"].sum() != len(valid_cooperation):
    raise AssertionError(
        "Aggregated cooperation counts do not match the retained raw rows."
    )


# ============================================================
# 4. Download or reuse a frozen World Bank classification snapshot
# ============================================================

WB_TO_ANALYSIS_NAMES = {
    "Bahamas, The": "Bahamas",
    "Congo, Dem. Rep.": "DR Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Egypt, Arab Rep.": "Egypt",
    "Gambia, The": "Gambia",
    "Iran, Islamic Rep.": "Iran",
    "Korea, Dem. People's Rep.": "North Korea",
    "Korea, Rep.": "South Korea",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Lao PDR": "Laos",
    "Micronesia, Fed. Sts.": "Micronesia",
    "Moldova": "Moldova",
    "Naoero": "Nauru",
    "Russian Federation": "Russia",
    "Slovak Republic": "Slovakia",
    "Somalia, Fed. Rep.": "Somalia",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "Syrian Arab Republic": "Syria",
    "Tanzania": "Tanzania",
    "Turkiye": "Türkiye",
    "United States": "United States of America",
    "Venezuela, RB": "Venezuela",
    "Viet Nam": "Vietnam",
    "Yemen, Rep.": "Yemen",
}

if WB_SNAPSHOT_FILE.exists():
    wb_characteristics = pd.read_csv(
        WB_SNAPSHOT_FILE,
        encoding="utf-8",
    )
    snapshot_source = "existing frozen snapshot"
else:
    with urllib.request.urlopen(WB_API_URL, timeout=60) as response:
        wb_payload = json.load(response)

    wb_raw = pd.json_normalize(wb_payload[1])
    wb_raw = wb_raw.loc[
        wb_raw["region.value"].ne("Aggregates")
    ].copy()

    wb_raw["Country_Analysis"] = (
        wb_raw["name"]
        .replace(WB_TO_ANALYSIS_NAMES)
        .map(normalize_country)
    )

    retrieved_utc = datetime.now(timezone.utc).isoformat()

    wb_characteristics = (
        wb_raw[
            [
                "Country_Analysis",
                "id",
                "name",
                "incomeLevel.value",
                "region.value",
            ]
        ]
        .rename(
            columns={
                "id": "World_Bank_ISO3",
                "name": "World_Bank_Name",
                "incomeLevel.value": "Income_Group",
                "region.value": "Region",
            }
        )
        .copy()
    )

    wb_characteristics["Income_Group"] = (
        wb_characteristics["Income_Group"].astype(str).str.strip()
    )

    wb_characteristics["Region"] = (
        wb_characteristics["Region"].astype(str).str.strip()
    )

    wb_characteristics["Snapshot_Retrieved_UTC"] = retrieved_utc
    wb_characteristics["Source_URL"] = WB_API_URL

    wb_characteristics = (
        wb_characteristics.loc[
            wb_characteristics["Country_Analysis"].isin(universe_set)
        ]
        .drop_duplicates(subset="Country_Analysis", keep="first")
        .sort_values("Country_Analysis")
        .reset_index(drop=True)
    )

    wb_characteristics.to_csv(
        WB_SNAPSHOT_FILE,
        index=False,
        encoding="utf-8",
    )

    snapshot_source = "new World Bank API snapshot"

required_characteristic_columns = {
    "Country_Analysis",
    "Income_Group",
    "Region",
}

if not required_characteristic_columns.issubset(wb_characteristics.columns):
    raise ValueError(
        "World Bank snapshot is missing required columns: "
        + ", ".join(sorted(required_characteristic_columns))
    )

wb_characteristics["Country_Analysis"] = (
    wb_characteristics["Country_Analysis"].map(normalize_country)
)

wb_characteristics["Income_Group"] = (
    wb_characteristics["Income_Group"].astype(str).str.strip()
)

wb_characteristics["Region"] = (
    wb_characteristics["Region"].astype(str).str.strip()
)

wb_characteristics = (
    wb_characteristics.loc[
        wb_characteristics["Country_Analysis"].isin(universe_set)
    ]
    .drop_duplicates(subset="Country_Analysis", keep="first")
    .copy()
)

missing_wb_countries = sorted(
    universe_set - set(wb_characteristics["Country_Analysis"])
)

if missing_wb_countries:
    raise ValueError(
        "Missing World Bank classifications for: "
        + ", ".join(missing_wb_countries)
    )

invalid_wb_rows = wb_characteristics.loc[
    wb_characteristics["Income_Group"].isin(["Aggregates", "Not classified"])
    | wb_characteristics["Region"].isin(["Aggregates", "Not classified"])
]

if not invalid_wb_rows.empty:
    raise ValueError(
        "Invalid World Bank classifications for: "
        + ", ".join(sorted(invalid_wb_rows["Country_Analysis"]))
    )

controlled_data = master.merge(
    wb_characteristics[
        ["Country_Analysis", "Income_Group", "Region"]
    ],
    on="Country_Analysis",
    how="left",
    validate="one_to_one",
)

if controlled_data[["Income_Group", "Region"]].isna().any().any():
    missing = controlled_data.loc[
        controlled_data[["Income_Group", "Region"]].isna().any(axis=1),
        "Country_Analysis",
    ].tolist()
    raise ValueError(
        "Missing characteristics after merge for: "
        + ", ".join(missing)
    )


# ============================================================
# 5. Baseline and partial Spearman correlations
# ============================================================

def partial_spearman(
    data: pd.DataFrame,
    predictor: str,
    outcome: str,
    controls: list[str],
) -> dict[str, object]:
    """Partial Spearman via correlations of rank-regression residuals."""
    required_columns = [predictor, outcome, *controls]
    working = data[required_columns].dropna().copy()

    predictor_rank = rankdata(
        working[predictor],
        method="average",
    )

    outcome_rank = rankdata(
        working[outcome],
        method="average",
    )

    control_dummies = pd.get_dummies(
        working[controls],
        drop_first=True,
        dtype=float,
    )

    control_matrix = np.column_stack(
        [
            np.ones(len(working)),
            control_dummies.to_numpy(),
        ]
    )

    predictor_coefficients = np.linalg.lstsq(
        control_matrix,
        predictor_rank,
        rcond=None,
    )[0]

    outcome_coefficients = np.linalg.lstsq(
        control_matrix,
        outcome_rank,
        rcond=None,
    )[0]

    predictor_residuals = (
        predictor_rank
        - control_matrix @ predictor_coefficients
    )

    outcome_residuals = (
        outcome_rank
        - control_matrix @ outcome_coefficients
    )

    partial_rho = float(
        np.corrcoef(
            predictor_residuals,
            outcome_residuals,
        )[0, 1]
    )

    number_of_controls = int(
        np.linalg.matrix_rank(control_matrix) - 1
    )

    degrees_of_freedom = (
        len(working) - number_of_controls - 2
    )

    if degrees_of_freedom <= 0:
        raise ValueError("Insufficient degrees of freedom.")

    t_statistic = (
        partial_rho
        * np.sqrt(
            degrees_of_freedom
            / (1 - partial_rho**2)
        )
    )

    p_value = float(
        2 * t.sf(abs(t_statistic), df=degrees_of_freedom)
    )

    return {
        "N": len(working),
        "Number_of_Dummy_Controls": number_of_controls,
        "Spearman_rho": partial_rho,
        "Degrees_of_Freedom": degrees_of_freedom,
        "p_value_two_sided": p_value,
        "Significant_at_05": bool(p_value < 0.05),
    }


outcomes = [
    "Cooperation_Entries",
    "Cooperation_Breadth",
]

baseline_rows = []

for outcome in outcomes:
    test = spearmanr(
        controlled_data["Documented_Support"],
        controlled_data[outcome],
        alternative="two-sided",
        nan_policy="raise",
    )

    baseline_rows.append(
        {
            "Model": "Uncontrolled baseline",
            "Controls": "None",
            "Outcome": outcome,
            "N": len(controlled_data),
            "Number_of_Dummy_Controls": 0,
            "Spearman_rho": float(test.statistic),
            "Degrees_of_Freedom": len(controlled_data) - 2,
            "p_value_two_sided": float(test.pvalue),
            "Significant_at_05": bool(test.pvalue < 0.05),
        }
    )

baseline_results = pd.DataFrame(baseline_rows)

control_models = {
    "Income group": ["Income_Group"],
    "Geographic region": ["Region"],
    "Income group + region": ["Income_Group", "Region"],
}

controlled_rows = []

for model_name, controls in control_models.items():
    for outcome in outcomes:
        test = partial_spearman(
            data=controlled_data,
            predictor="Documented_Support",
            outcome=outcome,
            controls=controls,
        )

        controlled_rows.append(
            {
                "Model": model_name,
                "Controls": " + ".join(controls),
                "Outcome": outcome,
                **test,
            }
        )

controlled_results = pd.DataFrame(controlled_rows)

all_results = pd.concat(
    [baseline_results, controlled_results],
    ignore_index=True,
)

descriptive_statistics = (
    controlled_data
    .groupby("Support_Status", as_index=False)
    .agg(
        N=("Country_Analysis", "size"),
        Mean_Entries=("Cooperation_Entries", "mean"),
        Median_Entries=("Cooperation_Entries", "median"),
        Mean_Breadth=("Cooperation_Breadth", "mean"),
        Median_Breadth=("Cooperation_Breadth", "median"),
    )
)

income_group_counts = (
    controlled_data["Income_Group"]
    .value_counts()
    .rename_axis("Income_Group")
    .reset_index(name="N")
)

region_counts = (
    controlled_data["Region"]
    .value_counts()
    .rename_axis("Region")
    .reset_index(name="N")
)


# ============================================================
# 6. Save analytical outputs
# ============================================================

controlled_data.to_csv(
    OUTPUT_DIR / "country_level_analysis.csv",
    index=False,
    encoding="utf-8",
)

descriptive_statistics.to_csv(
    OUTPUT_DIR / "descriptive_statistics_by_support.csv",
    index=False,
    encoding="utf-8",
)

baseline_results.to_csv(
    OUTPUT_DIR / "baseline_spearman_results.csv",
    index=False,
    encoding="utf-8",
)

controlled_results.to_csv(
    OUTPUT_DIR / "partial_spearman_controlled_results.csv",
    index=False,
    encoding="utf-8",
)

all_results.to_csv(
    OUTPUT_DIR / "all_spearman_results.csv",
    index=False,
    encoding="utf-8",
)

income_group_counts.to_csv(
    OUTPUT_DIR / "income_group_counts.csv",
    index=False,
    encoding="utf-8",
)

region_counts.to_csv(
    OUTPUT_DIR / "region_counts.csv",
    index=False,
    encoding="utf-8",
)

excluded_cooperation.to_csv(
    OUTPUT_DIR / "excluded_non_un_cooperation_rows.csv",
    index=False,
    encoding="utf-8",
)


# ============================================================
# 7. Visualization 1: country-level distributions
# ============================================================

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.edgecolor": "#30343B",
        "axes.labelcolor": "#30343B",
        "xtick.color": "#30343B",
        "ytick.color": "#30343B",
    }
)

distribution_specs = [
    {
        "column": "Cooperation_Entries",
        "title": "Cooperation entries",
        "ylabel": "Number of cooperation entries",
    },
    {
        "column": "Cooperation_Breadth",
        "title": "Cooperation breadth",
        "ylabel": "Number of distinct cooperation areas",
    },
]

group_order = [0, 1]
group_labels = {
    0: "No documented\nsupport",
    1: "Documented\nsupport",
}
group_colors = {0: "#778795", 1: "#C75C3A"}
rng = np.random.default_rng(20260807)

fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.9))
fig.patch.set_facecolor("white")

for ax, specification in zip(axes, distribution_specs):
    outcome = specification["column"]
    grouped_values = [
        controlled_data.loc[
            controlled_data["Documented_Support"].eq(group),
            outcome,
        ].to_numpy()
        for group in group_order
    ]

    violins = ax.violinplot(
        grouped_values,
        positions=[0, 1],
        widths=0.72,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        bw_method=0.45,
    )

    for group, body in zip(group_order, violins["bodies"]):
        body.set_facecolor(group_colors[group])
        body.set_edgecolor(group_colors[group])
        body.set_alpha(0.20)

    boxes = ax.boxplot(
        grouped_values,
        positions=[0, 1],
        widths=0.20,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "white", "linewidth": 2.2},
        whiskerprops={"color": "#4B515B", "linewidth": 1.2},
        capprops={"color": "#4B515B", "linewidth": 1.2},
    )

    for group, box in zip(group_order, boxes["boxes"]):
        box.set_facecolor(group_colors[group])
        box.set_edgecolor("#4B515B")

    for position, (group, values) in enumerate(
        zip(group_order, grouped_values)
    ):
        jitter = rng.normal(
            loc=position,
            scale=0.055,
            size=len(values),
        )

        ax.scatter(
            jitter,
            values,
            s=25,
            color=group_colors[group],
            alpha=0.55,
            edgecolor="white",
            linewidth=0.45,
            zorder=3,
        )

    result = baseline_results.loc[
        baseline_results["Outcome"].eq(outcome)
    ].iloc[0]

    p_text = (
        "p < 0.001"
        if result["p_value_two_sided"] < 0.001
        else f"p = {result['p_value_two_sided']:.3f}"
    )

    ax.text(
        0.04,
        0.95,
        f"Spearman $\\rho$ = {result['Spearman_rho']:.3f}\n{p_text}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10.5,
        color="#30343B",
        bbox={
            "boxstyle": "round,pad=0.45",
            "facecolor": "white",
            "edgecolor": "#D8DDE3",
            "alpha": 0.96,
        },
    )

    counts = [len(values) for values in grouped_values]
    ax.set_xticks([0, 1])
    ax.set_xticklabels(
        [
            f"{group_labels[group]}\n(n = {count})"
            for group, count in zip(group_order, counts)
        ]
    )
    ax.set_title(
        specification["title"],
        fontsize=13,
        fontweight="bold",
        loc="left",
        pad=13,
    )
    ax.set_ylabel(specification["ylabel"])
    ax.set_xlim(-0.55, 1.55)
    ax.set_ylim(bottom=-0.55)
    ax.grid(axis="y", color="#E7EAEE", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "Documented UNSC support and science-diplomacy cooperation",
    x=0.07,
    y=0.975,
    ha="left",
    fontsize=16,
    fontweight="bold",
    color="#22262D",
)

fig.text(
    0.07,
    0.91,
    "Complete sample of 192 non-India UN members; two-sided Spearman tests",
    ha="left",
    fontsize=10.5,
    color="#606873",
)

fig.text(
    0.07,
    0.025,
    "Points are countries; boxes show median and IQR. Zero means no cooperation was recorded in the supplied data.",
    ha="left",
    fontsize=9,
    color="#606873",
)

fig.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.18,
    top=0.82,
    wspace=0.30,
)

fig.savefig(
    OUTPUT_DIR / "baseline_support_cooperation_distributions.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    OUTPUT_DIR / "baseline_support_cooperation_distributions.svg",
    bbox_inches="tight",
    facecolor="white",
)

plt.close(fig)


# ============================================================
# 8. Visualization 2: baseline versus controlled estimates
# ============================================================

model_order = [
    "Uncontrolled baseline",
    "Income group",
    "Geographic region",
    "Income group + region",
]

outcome_specs = [
    ("Cooperation_Entries", "Cooperation entries", "#2F6B8A"),
    ("Cooperation_Breadth", "Cooperation breadth", "#C65D3A"),
]


def format_p_value(value: float) -> str:
    if value < 0.001:
        return "p < 0.001"
    return f"p = {value:.3f}"


fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.0, 6.2),
    sharey=True,
)

fig.patch.set_facecolor("white")
y_positions = np.arange(len(model_order))

for ax, (outcome, panel_title, color) in zip(axes, outcome_specs):
    plot_data = (
        all_results.loc[all_results["Outcome"].eq(outcome)]
        .set_index("Model")
        .loc[model_order]
        .reset_index()
    )

    estimates = plot_data["Spearman_rho"].to_numpy()

    for y_position in y_positions:
        if y_position % 2 == 0:
            ax.axhspan(
                y_position - 0.42,
                y_position + 0.42,
                color="#F4F6F8",
                zorder=0,
            )

    ax.hlines(
        y=y_positions,
        xmin=0,
        xmax=estimates,
        color=color,
        linewidth=3,
        alpha=0.75,
        zorder=2,
    )

    ax.scatter(
        estimates,
        y_positions,
        s=115,
        color=color,
        edgecolor="white",
        linewidth=1.2,
        zorder=3,
    )

    for y_position, row in zip(y_positions, plot_data.itertuples()):
        ax.text(
            row.Spearman_rho + 0.012,
            y_position,
            (
                f"$\\rho$ = {row.Spearman_rho:.3f}\n"
                f"{format_p_value(row.p_value_two_sided)}"
            ),
            ha="left",
            va="center",
            fontsize=9.2,
            color="#30343B",
        )

    ax.axvline(
        0,
        color="#7B828C",
        linestyle=(0, (4, 3)),
        linewidth=1.2,
        zorder=1,
    )

    ax.set_title(
        panel_title,
        loc="left",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("Spearman correlation ($\\rho$)")
    ax.set_xlim(-0.02, 0.54)
    ax.set_xticks(np.arange(0, 0.51, 0.1))
    ax.grid(axis="x", color="#E1E5E9", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)

axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(model_order)
axes[0].invert_yaxis()

fig.suptitle(
    "The association after controlling for income and geography",
    x=0.08,
    y=0.975,
    ha="left",
    fontsize=16,
    fontweight="bold",
    color="#22262D",
)

fig.text(
    0.08,
    0.915,
    "Baseline and partial Spearman correlations for 192 non-India UN members",
    ha="left",
    fontsize=10.5,
    color="#606873",
)

fig.text(
    0.08,
    0.025,
    "Points show correlation estimates; p-values are two-sided. Dashed line = no association.",
    ha="left",
    fontsize=9,
    color="#606873",
)

fig.subplots_adjust(
    left=0.22,
    right=0.98,
    bottom=0.16,
    top=0.82,
    wspace=0.22,
)

fig.savefig(
    OUTPUT_DIR / "income_region_controlled_correlations.png",
    dpi=400,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    OUTPUT_DIR / "income_region_controlled_correlations.svg",
    bbox_inches="tight",
    facecolor="white",
)

plt.close(fig)


# ============================================================
# 9. Validation report and console summary
# ============================================================

validation_rows = [
    {
        "Check": "Support input filename",
        "Value": SUPPORT_FILENAME,
        "Status": "PASS",
    },
    {
        "Check": "Cooperation input filename",
        "Value": COOPERATION_FILENAME,
        "Status": "PASS",
    },
    {
        "Check": "Support input SHA256",
        "Value": file_sha256(support_path),
        "Status": "PASS",
    },
    {
        "Check": "Cooperation input SHA256",
        "Value": file_sha256(cooperation_path),
        "Status": "PASS",
    },
    {
        "Check": "Raw support rows",
        "Value": len(support),
        "Status": "PASS" if len(support) == 79 else "REVIEW",
    },
    {
        "Check": "Raw cooperation rows",
        "Value": len(cooperation),
        "Status": "PASS" if len(cooperation) == 805 else "REVIEW",
    },
    {
        "Check": "Retained UN-member cooperation rows",
        "Value": len(valid_cooperation),
        "Status": "PASS",
    },
    {
        "Check": "Excluded non-UN or unknown rows",
        "Value": len(excluded_cooperation),
        "Status": "PASS",
    },
    {
        "Check": "Countries in analysis",
        "Value": len(controlled_data),
        "Status": "PASS" if len(controlled_data) == 192 else "FAIL",
    },
    {
        "Check": "Documented supporters",
        "Value": int(controlled_data["Documented_Support"].sum()),
        "Status": "PASS",
    },
    {
        "Check": "Countries with active cooperation",
        "Value": int((controlled_data["Cooperation_Entries"] > 0).sum()),
        "Status": "PASS",
    },
    {
        "Check": "World Bank classification coverage",
        "Value": int(
            controlled_data[["Income_Group", "Region"]]
            .notna()
            .all(axis=1)
            .sum()
        ),
        "Status": "PASS",
    },
    {
        "Check": "World Bank snapshot source",
        "Value": snapshot_source,
        "Status": "PASS",
    },
]

validation_summary = pd.DataFrame(validation_rows)

validation_summary.to_csv(
    OUTPUT_DIR / "validation_summary.csv",
    index=False,
    encoding="utf-8",
)

summary_lines = [
    "CLEAN INCOME- AND REGION-CONTROLLED SPEARMAN ANALYSIS",
    "",
    f"Support input: {SUPPORT_FILENAME}",
    f"Cooperation input: {COOPERATION_FILENAME}",
    f"Output folder: {OUTPUT_DIR}",
    f"Countries: {len(controlled_data)}",
    f"Documented supporters: {int(controlled_data['Documented_Support'].sum())}",
    f"Active cooperation countries: {int((controlled_data['Cooperation_Entries'] > 0).sum())}",
    f"Valid cooperation rows: {len(valid_cooperation)}",
    f"Excluded non-UN/unknown rows: {len(excluded_cooperation)}",
    "",
    "RESULTS",
]

for row in all_results.itertuples():
    summary_lines.append(
        f"{row.Model} | {row.Outcome} | N={row.N} | "
        f"rho={row.Spearman_rho:.6f} | "
        f"p={row.p_value_two_sided:.8f}"
    )

(OUTPUT_DIR / "analysis_summary.txt").write_text(
    "\n".join(summary_lines) + "\n",
    encoding="utf-8",
)

print("\nClean analysis completed successfully.")
print(f"All outputs saved in: {OUTPUT_DIR}")
print("\nSpearman results:")
print(
    all_results[
        [
            "Model",
            "Outcome",
            "N",
            "Spearman_rho",
            "p_value_two_sided",
            "Significant_at_05",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nIncome-group counts:")
print(income_group_counts.to_string(index=False))

print("\nRegion counts:")
print(region_counts.to_string(index=False))